# Audit the archived baseline and ablation files.

Recorded-run workflow. Configure paths in `settings.json`; see the root README for dependencies.


In [ ]:
from pathlib import Path
import sys

_release = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "release_runtime.py").is_file()), None)
if _release is None:
    raise FileNotFoundError("Open this notebook from the extracted Code_Release_Candidate folder.")
sys.path.insert(0, str(_release))
from release_runtime import activate_project, load_json, load_text
PROJECT_ROOT, RELEASE_CLASSICAL_PYTHON, RELEASE_TORCH_PYTHON = activate_project()


In [ ]:
from pathlib import Path

REVISION_ROOT = Path("revision_outputs").expanduser().resolve()
DATA_DIR = Path("data").expanduser().resolve()

print("Kayıtlı çıktılar:", REVISION_ROOT)
print("Ham veriler:", DATA_DIR)


In [ ]:

import csv
import hashlib
import importlib.metadata
import json
import math
import platform
import sys
import zipfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path


def canonical_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'),
                                     ensure_ascii=False, allow_nan=False).encode()).hexdigest()


def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def read_json(path):
    with path.open(encoding='utf-8') as f:
        return json.load(f)


def file_check(path, expected=None):
    result = {'path': str(path), 'exists': path.is_file()}
    if not result['exists']:
        result['status'] = 'MISSING'
        return result
    try:
        result['bytes'] = path.stat().st_size
        result['sha256'] = digest(path)
        result['status'] = ('HASH_MATCH' if result['sha256'] == expected else 'HASH_MISMATCH') if expected else 'PRESENT_NO_REFERENCE_HASH'
    except (OSError, ValueError) as e:
        result.update(status='READ_ERROR', error=str(e))
    return result


def safe_child(base, relative):
    path = (base / relative).resolve()
    if not path.is_relative_to(base.resolve()):
        raise ValueError('Manifest path escapes its run directory: ' + relative)
    return path


def primary_key(row, stage):
    prefix = (row.get('dataset'), row.get('preprocessing')) if stage == '10' else ()
    return prefix + (row.get('representation'), row['method'], int(row['seed']), int(row['fold']))


def expected_jobs(stage, summary):
    for seed in summary['seeds']:
        for fold in summary['folds']:
            suffix = f'seed_{seed}/fold_{fold:02d}'
            if stage == '05':
                yield suffix, {'seed': seed, 'fold': fold, 'representation': 'AE_latent21'}
            elif stage in ['06', '07']:
                configs = summary['new_configurations'] if stage == '06' else summary['configurations']
                for cfg in configs:
                    yield cfg['name'] + '/' + suffix, {'seed': seed, 'fold': fold, 'representation': cfg['name']}
            else:
                for block in summary['analysis_blocks']:
                    for rep in summary['representations']:
                        identity = {**block, 'seed': seed, 'fold': fold, 'representation': rep}
                        yield f"{block['dataset']}/{block['preprocessing']}/{suffix}/{rep}", identity


def critical_names(stage, rep):
    common = ['selection_trials.json']
    if stage == '05':
        return common + ['encoder.keras', 'autoencoder.keras', 'scaler_parameters.json', 'pilot_plan.json', 'representation_metrics.csv']
    if stage == '06':
        return common + ['encoder.keras', 'autoencoder.keras', 'scaler_parameters.json', 'metrics.csv']
    if stage == '07':
        return common + ['pca_parameters.npz', 'scaler_parameters.json', 'metrics.csv']
    names = common + ['preprocessor.joblib', 'preprocessing_parameters.json', 'representation_info.json', 'metrics.csv']
    if rep.startswith('AE_'):
        names += ['encoder.keras', 'autoencoder.keras', 'latent_scaler.joblib']
    elif rep.startswith('PCA_'):
        names += ['representation.joblib']
    return names


def audit_job(run, stage, relative, identity, aggregate_index, plan_hash):
    folder = safe_child(run, relative)
    marker = folder / ('fold_summary.json' if stage == '05' else 'result.json')
    result = {'relative_directory': relative, 'identity': identity, 'issues': [], 'files': []}
    try:
        record = read_json(marker)
    except (OSError, ValueError) as e:
        result.update(status='MISSING_OR_UNREADABLE_RECORD', error=str(e))
        return result
    if record.get('status') != 'PASS':
        result['issues'].append('Historical job status is not PASS')
    if stage == '10':
        if record.get('identity') != identity:
            result['issues'].append('Job identity mismatch')
    else:
        if any(record.get(k) != identity[k] for k in ['seed', 'fold']):
            result['issues'].append('Fold/seed identity mismatch')
        if stage in ['06', '07'] and record.get('config', {}).get('name') != identity['representation']:
            result['issues'].append('Representation identity mismatch')
    if stage != '05' and record.get('plan_sha256') != plan_hash:
        result['issues'].append('Plan hash mismatch')
    rows = record.get('primary_results', []) if stage == '05' else [r for r in record.get('metrics', []) if r.get('RUL_window') == 30]
    expected_methods = {'IF', 'LOF', 'kNN'} | ({'MSE'} if stage in ['05', '06'] or identity['representation'].startswith('AE_') else set())
    expected_row_count = 7 if stage == '05' else len(expected_methods)
    if len(rows) != expected_row_count or {r.get('method') for r in rows} != expected_methods:
        result['issues'].append('Historical three-detector/MSE coverage mismatch')
    seen = set()
    for row in rows:
        try:
            key = primary_key(row, stage)
            if key in seen or key not in aggregate_index:
                raise ValueError('Missing/duplicate primary row')
            seen.add(key)
            baseline = aggregate_index[key]
            for field in ['AP', 'ROC_AUC', 'precision', 'recall', 'F1', 'Macro_F1', 'TP', 'FP', 'TN', 'FN', 'threshold', 'selection_AP_primary']:
                if not math.isclose(float(row[field]), float(baseline[field]), rel_tol=0, abs_tol=1e-12):
                    raise ValueError('Mismatch in ' + field)
            if row.get('selected_params') != baseline.get('selected_params'):
                raise ValueError('Selected parameters mismatch')
        except (KeyError, TypeError, ValueError) as e:
            result['issues'].append('Aggregate comparison: ' + str(e))
    manifest = record.get('output_sha256', {})
    for name in critical_names(stage, identity['representation']):
        check = file_check(folder / name, manifest.get(name))
        if check['status'] != 'HASH_MATCH':
            result['issues'].append(name + ': ' + check['status'])

        if check['status'] == 'HASH_MATCH' and name.endswith(('.keras', '.npz')):
            try:
                with zipfile.ZipFile(folder / name) as archive:
                    bad = archive.testzip()
                    names = set(archive.namelist())
                check['archive_crc_ok'] = bad is None
                if bad is not None or (name.endswith('.keras') and not {'config.json', 'metadata.json', 'model.weights.h5'}.issubset(names)):
                    result['issues'].append(name + ': invalid archive contents')
            except (OSError, ValueError, zipfile.BadZipFile, RuntimeError) as e:
                check['archive_error'] = str(e)
                result['issues'].append(name + ': unreadable archive')
        result['files'].append(check)

    prediction_names = ['outer_predictions.csv.gz', 'selection_scores.csv.gz'] if stage == '10' else ['outer_predictions.csv']
    result['prediction_files'] = [{'name': n, 'exists': (folder / n).is_file(), 'bytes': (folder / n).stat().st_size if (folder / n).is_file() else None, 'verified': 'presence_only'} for n in prediction_names]
    result['status'] = 'FILES_AND_RECORDED_METRICS_MATCH' if not result['issues'] else 'REVIEW_REQUIRED'
    return result


def audit_stage(root, spec):
    stage = spec['stage']
    folder = root / spec['directory']
    candidates = sorted(folder.rglob(spec['summary_name'])) if folder.is_dir() else []
    matches, discovered = [], []
    for path in candidates:
        item = {'path': str(path)}
        try:
            summary = read_json(path)
            item['canonical_summary_sha256'] = canonical_hash(summary)
            item['matches_supplied_summary'] = item['canonical_summary_sha256'] == spec['summary_canonical_sha256']
            if item['matches_supplied_summary']:
                matches.append((path, summary))
        except (OSError, ValueError) as e:
            item['error'] = str(e)
        discovered.append(item)
    result = {'stage': stage, 'summary_candidates': discovered, 'matching_runs': []}
    if not matches:
        result['status'] = 'SUPPLIED_SUMMARY_NOT_FOUND'
        return result
    for path, summary in matches:
        run = path.parent
        rows = summary['fold_primary_results']
        index = {primary_key(row, stage): row for row in rows}
        if len(index) != len(rows):
            raise ValueError('Duplicate primary keys in supplied summary')
        jobs = []
        expected = list(expected_jobs(stage, summary))
        print(f"{stage}: {run.name} — {len(expected)} kayıt kontrol ediliyor.", flush=True)
        for i, (relative, identity) in enumerate(expected, 1):
            jobs.append(audit_job(run, stage, relative, identity, index, summary.get('plan_sha256')))
            if i % 50 == 0 or i == len(expected):
                print(f"  {i}/{len(expected)}", flush=True)
        result['matching_runs'].append({'run_directory': str(run), 'expected_jobs': len(expected), 'job_status_counts': dict(Counter(j['status'] for j in jobs)), 'observed_methods': sorted({r['method'] for r in rows}), 'missing_original_detectors': ['DBSCAN', 'OC-SVM'], 'jobs': jobs})
    result['status'] = 'MULTIPLE_MATCHING_RUNS_REVIEW_REQUIRED' if len(matches) > 1 else ('FILES_AND_RECORDED_METRICS_MATCH' if all(j['status'] == 'FILES_AND_RECORDED_METRICS_MATCH' for j in result['matching_runs'][0]['jobs']) else 'REVIEW_REQUIRED')
    return result


def audit_all(root, data_dir, specs):
    root, data_dir = Path(root).expanduser().resolve(), Path(data_dir).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f'revision_outputs klasörü bulunamadı: {root}. İkinci hücrede REVISION_ROOT yolunu düzeltin.')
    versions = {'python': platform.python_version(), 'platform': platform.platform(), 'executable': sys.executable}
    for package in ['numpy', 'pandas', 'scikit-learn', 'tensorflow', 'keras']:
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = 'NOT_INSTALLED'
    protocol_checks = []
    for item in specs['protocols']:
        for name, h in item['files'].items():
            protocol_checks.append(file_check(safe_child(root / item['directory'], name), h))
    data_checks = [file_check(data_dir / name, h) for name, h in specs['train_files'].items()]
    stage_checks = [audit_stage(root, item) for item in specs['stages']]
    inputs_ok = all(r['status'] == 'HASH_MATCH' for r in protocol_checks + data_checks)
    files_ok = all(r['status'] == 'FILES_AND_RECORDED_METRICS_MATCH' for r in stage_checks)
    return {'stage': '02B_five_method_recovery_audit', 'created_at_utc': datetime.now(timezone.utc).isoformat(), 'status': 'READY_FOR_MODEL_LOAD_AND_SCORE_REPRODUCTION_CHECK' if inputs_ok and files_ok else 'REVIEW_REQUIRED', 'revision_root': str(root), 'data_directory': str(data_dir), 'versions': versions, 'original_detectors': ['DBSCAN', 'IF', 'LOF', 'OC-SVM', 'kNN'], 'model_training_performed': False, 'model_deserialization_performed': False, 'performance_metrics_recomputed_from_predictions': False, 'new_DBSCAN_or_OCSVM_results_produced': False, 'previous_outputs_modified': False, 'protocol_checks': protocol_checks, 'training_data_checks': data_checks, 'stages': stage_checks, 'limits': ['File integrity and stored summary agreement do not prove model loadability or score reproducibility.', 'Large prediction files were checked for presence/size only.', 'No five-detector protocol or DBSCAN extension was silently adopted.', 'No test dataset was opened.']}


def save_audit(report, root):
    folder = Path(root) / '02B_five_method_recovery_audit' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    folder.mkdir(parents=True, exist_ok=False)
    path = folder / 'recovery_audit_summary.json'
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    rows = []
    for stage in report['stages']:
        for run in stage['matching_runs']:
            for job in run['jobs']:
                rows.append({'stage': stage['stage'], 'run_directory': run['run_directory'], 'job': job['relative_directory'], 'status': job['status'], 'issues': ' | '.join(job.get('issues', [])), 'error': job.get('error', '')})
    with (folder / 'job_inventory.csv').open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=['stage', 'run_directory', 'job', 'status', 'issues', 'error'])
        writer.writeheader()
        writer.writerows(rows)
    print('\nDENETİM SONUCU:', report['status'])
    for stage in report['stages']:
        print(stage['stage'], stage['status'])
        for run in stage['matching_runs']:
            print(' ', run['job_status_counts'])
    print('\nPaylaşacağınız dosya:', path.resolve())
    print('Bu bir performans deneyi değildir; DBSCAN/OC-SVM sonuçları henüz üretilmedi.')
    return path


In [ ]:
APPROVED_INPUTS = load_json('assets/02B/04_approved_inputs.json')

report = audit_all(REVISION_ROOT, DATA_DIR, APPROVED_INPUTS)
summary_path = save_audit(report, REVISION_ROOT)
